In [ ]:
# Summary: 
# Automate scraping pages of battles from PS, getting their respective replays, 
# and saving them to disk. The 'main' function is toward the end.

In [1]:
# Some basic imports
import numpy as np
import pandas as pd

import json, os, time
import logging, requests
import itertools
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
from urllib.parse import urlencode, urljoin

import re # for regex things

# Useful 'constants'
SEARCH_BASE_URL = "https://replay.pokemonshowdown.com/api/replays/search"
REPLAY_BASE_URL = "https://replay.pokemonshowdown.com/"
USER_BASE_URL = "https://pokemonshowdown.com/users/"

# Only needed if you want messages to appear when an error, etc. occurs
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

# Getting pages/replays
-----------

In [2]:
# Dataclasses
# ===============================================

# Roughly, we 
#    1. Pull pages full of Battles;
#    2. Get `battle_id`s from each Battle;
#    3. Pull and write the Replay using the `battle_id`.
# 
# The `Battle` dataclass is unnecessary at present, but we can use it later if needed.

@dataclass
class Battle:
    uploadtime: int
    id: str
    format: str
    players: list[str]
    rating: Optional[int] = None

@dataclass
class Replay:
    id: str
    format: str
    players: list[str]
    log: str
    uploadtime: int
    views: int
    formatid: str
    rating: object = None
    private: int = 0
    password: object = None

In [3]:
# ensuring directory exists
# ===============================================
def ensure_dir(name):
    try:
        Path(name).mkdir(exist_ok=True)
    except OSError as e:
        logger.error("failed to create dir %s: %s", name, e)
        raise SystemExit(1)

In [4]:
# %%%%%%%%% 'Getters' %%%%%%%%%%%%%%%%%%%%%%%%%%%
def get_page(page_num, fmt='') -> list[dict]:
    '''
    Returns a list of dicts, with each dict corresponding to a `json` of a battle.
    The dicts have keys/values like that of the Battle dataclass; `id` is the only crucial one.

    Leaving `fmt=''` searches for battles of any format.
    '''
    
    params = urlencode({"format": fmt, "page": str(page_num)}) 
    url = f"{SEARCH_BASE_URL}?{params}"

    # This deals with an apparent quirk in PS's search: the first-page results are fine, 
    # but subsequent searches seem to want this `Referer` in the GET request.
    headers = {}
    if page_num > 1: 
        headers = {"Referer": f"https://replay.pokemonshowdown.com/?format={fmt}&page={page_num}"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get page %d: %s", page_num, e)
        return []

    # The API response starts with a leading character ']' before valid JSON,
    # so remove it
    raw = response.content[1:]
    
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        logger.error("failed to parse page %d: %s", page_num, e)
        return []
    return data


# ===========================
def get_replay(battle_id: str) -> Replay:
    '''
    Returns a Replay object pulled using the unique battle_id.
    
    Much like `get_page` in function .
    '''
    
    url = REPLAY_BASE_URL + f'{battle_id}' + '.json'
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get replay: %s", e)
        return None

    try:
        data = response.json()
    except json.JSONDecodeError as e:
        logger.error("failed to parse replay: %s", e)
        return None

    return Replay(
        id=data.get("id", ""),
        format=data.get("format", ""),
        players=data.get("players", []),
        log=data.get("log", ""),
        uploadtime=data.get("uploadtime", 0),
        views=data.get("views", 0),
        formatid=data.get("formatid", ""),
        rating=data.get("rating"),
        private=data.get("private", 0),
        password=data.get("password"),
    )

# The 'main' scraper

In [5]:
# Definition
def scrape_replays(page_num, fmt=""):
    '''
    Search page, get battle Replays, and write to file.
    '''
    
    # main page-scraping
    page = get_page(page_num, fmt)
    
    if page == [] : 
        print(f"Failed to get page {page_num}.")
        return None
    
    for battle in page :
        # main replay-scraping
        replay = get_replay(battle['id'])
        if replay is None:
            logger.error("failed to get replay %s", battle['id'])
            return None
        
        # Writing
        if fmt == "" : 
            out_path = Path("replays") / f"{replay.id}.json"
        else : 
            ensure_dir(f'replays/{fmt}')
            out_path = Path(f"replays/{fmt}") / f"{replay.id}.json"
        
        try:
            out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
        except OSError as e:
            logger.error("failed to save replay %s : %s", battle['id'], e)

In [10]:
# Automation/Running

FORMAT='gen9-randombattle';
ensure_dir('./replays')
ensure_dir('./replays/'+FORMAT)

# NOTE: if this value is too small, the PS server (hosted on Cloudflare) 
# will 'cut-off' our scripts (with some unknown severity). Though I have 
# been told that 0.25 seconds 'ought to be fine', this 5 second delay 
# should be plenty fast, and have a wide safety margin.
PAUSE = 5; # seconds between pages

# Note: even trying searching manually, it seems results stop at page 100
for j in range(1,101): 
    print(f"Now working on page {j}.")
    
    scrape_replays(j, fmt=FORMAT)
    
    print(f"\tDone; taking a {PAUSE} second break.")
    time.sleep(PAUSE)

Now working on page 110.
Failed to get page 110.
	Done; taking a 5 second break.
Now working on page 111.
Failed to get page 111.
	Done; taking a 5 second break.
Now working on page 112.
Failed to get page 112.
	Done; taking a 5 second break.
Now working on page 113.
Failed to get page 113.
	Done; taking a 5 second break.


KeyboardInterrupt: 